## PGA Draftkings Notebook
Use [PGA Website](https://www.pgatour.com/tournaments/schedule.html) to look up tournament info and fill out first USER INPUT block below. (Fetch/XHR from Network tab, Paylod sub-tab)

Looks like the tournament ID is also in the address bar.

### User Input

In [1]:
# === USER INPUTS ===
# Old Tournament
old_tournament_name = "Biltmore Championship Asheville"
tournament_date = "9/20/2026"  # Ending date of tournament
old_course = "East Lake Golf Club"
tournament_id = "R2026557"  # Tournament ID from the PGA Tour API - Also visible in the URL of the tournament page

# New Tournament
new_tournament_name = "Bank of Utah Championship"
new_ending_date = "10/4/2026"
new_course = "Black Desert Resort"
new_season = 2026

# Test mode: set True to dry-run the workflow against a past week's config.
# Skips the odds DB write and relaxes the stale-config guard.
test_mode = False


# === LIBRARIES AND VARIABLES ===
# Import necessary libraries
import requests
import pandas as pd
from datetime import datetime
import sqlite3 as sql
import numpy as np
from numpy import nan
import os
import importlib
import utils.db_utils
from utils.db_utils import TOURNAMENT_NAME_MAP, PLAYER_NAME_MAP

# === TOURNAMENT CONFIG ===
tournament_config = {
    "old": {
        "name": old_tournament_name,
        "date": tournament_date,
        "course": old_course,
        "id": tournament_id
    },
    "new": {
        "name": new_tournament_name,
        "course": new_course,
        "season": new_season,
        "ending_date": pd.to_datetime(new_ending_date, format="%m/%d/%Y")
    }
}


### New Course / Tournament History Check

Exact-match history for `new_tournament_name` / `new_course` **before** running the rest of the notebook. Course-history joins are exact-string matches, so a tournament whose past rows show a differently spelled course (or an unexpectedly empty table) means fix the spelling first.

In [2]:
# === NEW COURSE / TOURNAMENT HISTORY CHECK ===
# Prior events exactly matching new_tournament_name or new_course. A misspelled
# course shows up as tournament rows whose COURSE differs from new_course; an
# empty table for an event that should have history means check the spelling.
with sql.connect("data/golf.db") as _conn:
    _events = pd.read_sql(
        "SELECT DISTINCT SEASON, ENDING_DATE, TOURNAMENT, COURSE FROM tournaments",
        _conn, parse_dates=["ENDING_DATE"])

_hist = (_events[(_events["TOURNAMENT"] == new_tournament_name) |
                 (_events["COURSE"] == new_course)]
         .sort_values("ENDING_DATE", ascending=False)
         .reset_index(drop=True))
print(f"new_tournament_name = {new_tournament_name!r}: "
      f"{(_hist['TOURNAMENT'] == new_tournament_name).sum()} prior events")
print(f"new_course = {new_course!r}: {(_hist['COURSE'] == new_course).sum()} prior events")
if _hist.empty:
    print("No exact matches on either name -- new event, or check spelling.")
_hist

new_tournament_name = 'Bank of Utah Championship': 1 prior events
new_course = 'Black Desert Resort': 2 prior events


,SEASON,ENDING_DATE,TOURNAMENT,COURSE
0,2025,2025-10-26,Bank of Utah Championship,Black Desert Resort
1,2024,2024-10-13,Black Desert Championship,Black Desert Resort


### Update Database

#### Old Tournament

In [3]:
importlib.reload(utils.db_utils)  # Only needed if you're actively editing db_utils.py
from utils.db_utils import update_tournament_results

# Change these each year!!
season = 2026
year = 20260  # Unique GraphQL year distinguishing number in case of multiple per year

# Run the update
db_path = "data/golf.db"  # Codex/PGA local copy
tournDf = update_tournament_results(tournament_config, db_path, season, year)

# Show just the most recent tournament added for confirmation
from sqlalchemy import create_engine

engine = create_engine(f"sqlite:///{db_path}")

query = f"""
SELECT *
FROM tournaments
WHERE TOURN_ID = '{tournament_config['old']['id']}'
  AND ENDING_DATE = '{datetime.strptime(tournament_config['old']['date'], '%m/%d/%Y').date()}'
"""

recent = pd.read_sql(query, engine)
engine.dispose()
recent.sort_values(by='FINAL_POS').head()

📦 Fetching results for tournament ID R2026557 (Biltmore Championship Asheville), year: 20260
ℹ️ Tournament 'Biltmore Championship Asheville' already exists — no new data inserted.


,SEASON,ENDING_DATE,TOURN_ID,TOURNAMENT,COURSE,PLAYER,POS,FINAL_POS,ROUNDS:1,ROUNDS:2,ROUNDS:3,ROUNDS:4,OFFICIAL_MONEY,FEDEX_CUP_POINTS
0,2026,2026-09-20,R2026557,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Jacob Bridgeman,1,1,-5,-4,-7,-10,"$900,000.00",0.000
1,2026,2026-09-20,R2026557,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Ben James,2,2,-4,-4,-6,-10,"$545,000.00",300.000
2,2026,2026-09-20,R2026557,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Ricky Castillo,T3,3,-2,-9,-6,-6,"$295,000.00",162.500
3,2026,2026-09-20,R2026557,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Neal Shipley,T3,3,-6,-9,-6,-2,"$295,000.00",162.500
4,2026,2026-09-20,R2026557,Biltmore Championship Asheville,The Cliffs at Walnut Cove,J.T. Poston,T5,5,-3,-5,-2,-11,"$193,125.00",0.000


#### Last Week's Report Card

Grades the predictions logged before last week's tournament against the results just imported above. The interactive per-player view lives in the dashboard's Prediction Tracker tab.

In [4]:
import importlib, utils.model
importlib.reload(utils.model)
from utils.model import grade_predictions

grade_predictions("data/golf.db")

📋 9 graded week(s) · avg top-15→top-20 hits: 7.67 (eval baseline ≈ 6.5)


,ENDING_DATE,TOURNAMENT,top15_in_top20,expected_top20,top15_cut_rate,status
0,2026-09-20,Biltmore Championship Asheville,7,5.5,0.67,graded
1,2026-08-30,TOUR Championship,11,6.5,1.00,graded
2,2026-08-23,BMW Championship,7,6.4,1.00,graded
3,2026-08-16,FedEx St. Jude Championship,10,6.7,1.00,graded
4,2026-08-09,Wyndham Championship,6,6.0,0.80,graded
5,2026-08-02,Rocket Classic,7,6.5,0.67,graded
6,2026-07-26,3M Open,5,5.8,0.67,graded
7,2026-07-19,The Open Championship,8,7.2,0.73,graded
8,2026-07-12,Genesis Scottish Open,8,7.1,0.67,graded


#### Stats

In [5]:
importlib.reload(utils.db_utils)
from utils.db_utils import update_season_stats  # <- This line is essential

# Change these each year!! 
statsYear = 2026

stats_df = update_season_stats(statsYear, db_path)
stats_df.head()

✅ Overwrote stats for season 2026 with 998 rows.


,PLAYER,SGTTG_RANK,SGTTG,SGOTT_RANK,SGOTT,SGAPR_RANK,SGAPR,SGATG_RANK,SGATG,SGP_RANK,...,DRIVING_DISTANCE,DRIVING_ACCURACY_RANK,DRIVING_ACCURACY,GIR_RANK,GIR,SCRAMBLING_RANK,SCRAMBLING,OWGR_RANK,OWGR,SEASON
0,A.J. Ewart,115.0,-0.240,128.0,-0.277,85.0,0.053,88.0,-0.016,8.0,...,302.8,106.0,58.13%,33.0,69.05%,29.0,63.33%,198,.7467,2026
1,Aaron Cockerill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,662,.1821,2026
2,Aaron Rai,37.0,0.525,82.0,0.077,27.0,0.385,63.0,0.063,92.0,...,293.3,4.0,71.30%,23.0,69.53%,124.0,57.06%,16,4.4265,2026
3,Aaron Wise,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,480,.2966,2026
4,Abraham Ancer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,208,.7229,2026


#### DB Maintenance (via Claude)

Weekly odds are captured automatically: the live-scrape cell in **Current Week** saves each week's odds to the database (`save_current_week_odds`), so season-long imports are no longer part of the routine.

One-off database maintenance is done by asking Claude (in this repo), which uses the helpers kept in `utils/db_utils.py`:

- **Backfill an odds gap** (missed a week, or a whole stretch of events): Claude runs `import_historical_odds(...)` against the golfodds.com archive pages.
- **Rename a player everywhere** (the usual fix — a DK/odds spelling differs from the DB canonical): Claude uses **`consolidate_player_name(db_path, wrong, correct, scope=...)`**. This is the canonical path: it renames the player across **every** table that has a PLAYER column — `odds`, `tournaments`, `stats`, `predictions`, and any table added later (they're discovered automatically, never hard-coded). Pass `scope="dk"` or `scope="player"` to also record the mapping in `name_mappings.json` so the incoming feed self-corrects going forward. It's an exact-match rename, so accented canonicals (Sebastián Muñoz, etc.) are left untouched. **Don't** fix a player one table at a time with `clean_player_names_in_table` — that's how `predictions` silently drifted out of sync (July 2026): a name got corrected in the results tables but not in the prediction log, which broke the Prediction Tracker's results join (Actual Pos showed blank).
- **Check for name drift**: `check_player_name_alignment(db_path)` flags any `predictions` name that matches no results table — the signature of a spelling mismatch. Run it after any DB name edit, or as a routine sanity check. Clean = every logged name lines up.
- **Fix name mismatches in the odds table only** (esp. tournament spelling, or duplicate-spelling collisions): `clean_odds_names(...)` updates the `odds` table in place. For a *player* rename prefer `consolidate_player_name` so the other tables come along too. Never drop/rebuild tables.
- **Course-name drift** (same venue, two spellings): Claude audits and renames to the most recent / PGA API spelling — the History Check cell near the top is the weekly tripwire.
- **Other tripwires**: the `odds coverage` printout in the training build and the per-week ✅/⚠️ checks. If coverage drops, ask Claude to audit and backfill.

The old manual cells (odds name cleanup, yearly archive import, scrape troubleshooting, Misc. Cleaning) were removed July 2026 — they live on in git history if ever needed.

# Model Pipeline (Pooled)

**Redesigned July 2026.** The model now trains on *every* tournament in the database
(point-in-time features, no lookahead) instead of only past editions of this week's
event, and outputs an **expected finish percentile blended with the betting market**
rather than a top-20 probability.

Forward-chained evaluation (2021–2025, 165 events; see `experiments/forward_eval.py`):

| pipeline | hits@15 | AUC |
|---|---|---|
| old per-event method | 5.92 | 0.696 |
| odds-only ranking | 6.38 | 0.722 |
| **this pipeline** | **6.49** | **0.728** |

Key columns in the output:
- **SCORE** — the optimizer input, 1 = best in field. Within-field average of the
  model's rank and the market's rank, rescaled to (0, 1].
- **MODEL_SCORE** — 1 − predicted finish percentile (model only, no market blend).
- **ODDS_SHARE** — vig-normalized implied win probability (the field sums to 1).
- **SG_FORM** — recency-weighted strokes gained per round vs the field
  (100-day halflife), computed from the round scores already in the database.

## Training Data

All events from 2016 through last week, each with features as they stood the
morning of that tournament. Takes ~1 minute. Odds coverage should be ≥95% —
if it drops, run the odds join audit (name mismatches are the usual cause).

In [6]:
import importlib
import utils.features, utils.model
importlib.reload(utils.features)
importlib.reload(utils.model)
from utils.model import build_pooled_training

db_path = "data/golf.db"

training_df, ctx = build_pooled_training(
    db_path,
    first_season=2016,
    as_of=tournament_config["new"]["ending_date"],
)
training_df.tail(3)

✅ 62318 training rows from 493 events (2016–2026), odds coverage 81%


,SEASON,ENDING_DATE,TOURNAMENT,COURSE,PLAYER,POS,FINAL_POS,FINISH_PCT,SGTTG_RANK,SGTTG,...,PCT_FORM_SHRUNK,COURSE_HISTORY,adj_ch,PCT_CH_SHRUNK,ODDS_SHARE,SG_FORM,SG_ROUNDS_12M,SG_CH_SHRUNK,TOP_20,FIELD_SIZE
62315,2026,2026-09-20,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Morgan Deneen,CUT,90,0.681818,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000232,NaN,NaN,NaN,0,132
62316,2026,2026-09-20,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Rasmus Hojgaard,W/D,90,0.681818,99.0,0.063,...,0.4108,NaN,NaN,NaN,0.022417,0.2715,70.0,NaN,0,132
62317,2026,2026-09-20,Biltmore Championship Asheville,The Cliffs at Walnut Cove,Lanto Griffin,W/D,90,0.681818,114.0,-0.058,...,0.5006,NaN,NaN,NaN,0.003457,-0.4491,53.0,NaN,0,132


## Current Week

### DraftKings Salaries

The field and its prices come straight from DraftKings now — there is no
`DKSalaries.csv` to download and drop into `data/` any more.

**It is two steps, and which one you skip IS the switch.**

| | what it does | at home | at work |
|---|---|---|---|
| **1. Download DraftKings salaries** | fetch this week's prices and save them | run it | **skip it** |
| **2. DraftKings Field** | build `dk` from the saved prices | run it | run it |

Your work network blocks `draftkings.com` outright, so step 1 dies in the TLS handshake
there. Skipping it is the intended workflow, not a workaround: **step 2 makes no request
to DraftKings at all** — not even a failed one — so nothing on that machine ever calls out.

The prices live in `data/salaries/dk-<season>-<ending date>-<tournament>.csv`, which is
committed to the repo. The handoff is a `git push` at home and a `git pull` at work.
**Commit both files step 1 writes** — the `.csv` and the `-meta.json` beside it.

**Only step 2 builds `dk`, on both machines, and that is the point.** At home it rebuilds
from the file step 1 has just saved, which costs no network and means the field you read
at work came out of the identical code path as the one you read at home.


#### 1. Download DraftKings salaries  ·  SKIP THIS AT WORK

**Safe to run again later in the week, and that is the reason to.** Withdrawals and late
commitments move the field all week; salaries barely move at all. An identical file
reports `unchanged` and touches nothing, and a differing one replaces it and says so.

It picks the draft group by **matching DraftKings' own tournament name against
`new_tournament_name`** from USER INPUTS — never "the first PGA TOUR one". The golf lobby
carries several tours at once (PGA TOUR, DP World Tour, Korn Ferry), each a plausible
field of real golfers at real salaries, so a wrong pick would be wrong about every player
while looking entirely normal. If nothing matches, it lists what is on offer and stops.


In [6]:
from utils import dk_api

# THE ONLY CELL THAT TALKS TO DRAFTKINGS.  *** SKIP IT AT WORK ***, where it
# fails in the TLS handshake - the firewall, not a bug. It leaves behind the
# two committed files step 2 reads:
#     data/salaries/dk-<season>-<ending date>-<tournament>.csv    the prices
#     data/salaries/dk-<season>-<ending date>-<tournament>-meta.json
# Neither can be re-fetched once the tournament has been played, so commit
# them as you go: DraftKings takes a contest down afterwards and there is no
# URL that returns last week's salaries.
#
# It reads new_tournament_name / new_season / new_ending_date out of
# tournament_config, so the file it writes is the file step 2 looks for.
# To pin one draft group instead (the lobby listing prints the ids):
#     dk_api.refresh_from_dk(tournament_config, draft_group=153546)

dk_api.refresh_from_dk(tournament_config)


  DraftKings lobby  3 salary-cap golf slate(s)
  dg154301  no field posted yet - skipped
  dg154300  no field posted yet - skipped
  dg154234  Bank of Utah Championship  -  Black Desert Resort
  first tee Thu 10/01 06:00AM ET   115 players priced   $6,000-$10,800

  data/salaries\dk-2026-2026-10-04-bank-of-utah-championship.csv   [written]
  commit data/salaries/ - DraftKings will not serve these again.


,Position,Name + ID,Name,ID,Roster Position,Salary,Game Info,TeamAbbrev,AvgPointsPerGame,Status
0,G,Jackson Koivun (44320059),Jackson Koivun,44320059,G,10800,Bank of Utah Championship,Golf,83.3,
1,G,Michael Brennan (44320060),Michael Brennan,44320060,G,10200,Bank of Utah Championship,Golf,66.6,
2,G,Benjamin James (44320061),Benjamin James,44320061,G,9900,Bank of Utah Championship,Golf,84.2,
3,G,Matt Wallace (44320062),Matt Wallace,44320062,G,9700,Bank of Utah Championship,Golf,73.2,
4,G,Maverick McNealy (44320063),Maverick McNealy,44320063,G,9600,Bank of Utah Championship,Golf,73.2,
...,...,...,...,...,...,...,...,...,...,...
110,G,Christo Lamprecht (44320168),Christo Lamprecht,44320168,G,6000,Bank of Utah Championship,Golf,40.2,
111,G,David Lingmerth (44320173),David Lingmerth,44320173,G,6000,Bank of Utah Championship,Golf,35.8,
112,G,Kensei Hirata (44320169),Kensei Hirata,44320169,G,6000,Bank of Utah Championship,Golf,42.8,
113,G,Marcelo Rozo (44320170),Marcelo Rozo,44320170,G,6000,Bank of Utah Championship,Golf,34.8,


#### 2. DraftKings Field  ·  ALWAYS RUN THIS

Reads the tournament's committed file and builds `dk` — the same `PLAYER` / `SALARY`
frame the hand-loaded CSV produced, names normalised to the database's spelling. It makes
**no request to DraftKings** under any circumstance and needs nothing step 1 left in
memory, so it is the same cell at home and at work.

**It looks the file up by `new_tournament_name` and `new_season`, not by "the newest
file".** A stale archive is the one failure this cannot fix for itself and the one that
does not look like a failure: last week's prices land on real players at plausible
salaries and every screen downstream looks ordinary. So if the configured tournament has
no saved file, this stops and says so rather than quietly serving the last one.

**Read the freshness line it prints** — when the fetch ran, and from which draft group.
The fix for a stale one is always the same: run step 1 at home, commit `data/salaries/`,
`git pull` here.


In [7]:
from utils import dk_api

# BUILD THE FIELD FROM THE SAVED PRICES. NO NETWORK, EVER - and it depends on
# nothing step 1 leaves in memory, deliberately, which is what makes "skip step
# 1 at work" a workflow rather than a trick.
#
# Identical to what `pd.read_csv("data/DKSalaries.csv")` produced by hand: the
# same two columns, the same DK_PLAYER_NAME_MAP, the same
# standardize_player_names. There is no DKSalaries.csv any more because it
# would be a second copy of the archive under a name that claims to be current
# - two files, both valid, free to disagree about which tournament is live.

dk = dk_api.load_field(tournament_config)
dk.head(10)


115 players in DK field   (saved prices, no network)
  data/salaries\dk-2026-2026-10-04-bank-of-utah-championship.csv
  fetched Mon 09/28 05:19PM ET (20h ago) from dg154234


,PLAYER,SALARY
0,Jackson Koivun,10800
1,Michael Brennan,10200
2,Ben James,9900
3,Matt Wallace,9700
4,Maverick McNealy,9600
5,Doug Ghim,9500
6,Marco Penge,9400
7,Jackson Suber,9300
8,Kevin Yu,9200
9,Davis Thompson,9100


### Vegas Odds (live scrape)

Scrapes this week's odds and **saves them into the odds table** so the current
season is fully priced for future training (no more waiting for the year-end
archive).
http://golfodds.com/weekly-odds.html

In [8]:
importlib.reload(utils.db_utils)
from utils.db_utils import get_current_week_odds
from utils.model import save_current_week_odds

odds_current = get_current_week_odds(
    season=tournament_config["new"]["season"],
    tournament_name=new_tournament_name,
)
save_current_week_odds(db_path, odds_current, tournament_config, dry_run=test_mode)
odds_current.head(10)

ℹ️ Odds page is serving: Bank of Utah Championship (ends 2026-10-04)
✅ Saved 123 odds rows for Bank of Utah Championship (2026-10-04)


,SEASON,TOURNAMENT,PLAYER,ODDS,VEGAS_ODDS
1,2026,Bank of Utah Championship,Jackson Koivun,12/1,12.0
2,2026,Bank of Utah Championship,Michael Brennan,16/1,16.0
3,2026,Bank of Utah Championship,Ben James,20/1,20.0
4,2026,Bank of Utah Championship,Maverick McNealy,20/1,20.0
5,2026,Bank of Utah Championship,Doug Ghim,25/1,25.0
6,2026,Bank of Utah Championship,Matt Wallace,25/1,25.0
7,2026,Bank of Utah Championship,Marco Penge,25/1,25.0
8,2026,Bank of Utah Championship,Jackson Suber,30/1,30.0
9,2026,Bank of Utah Championship,Davis Thompson,35/1,35.0
10,2026,Bank of Utah Championship,Kevin Yu,35/1,35.0


### Feature Rows

Same features as training, computed as of this week: prior-season stats, live
odds → market share, rolling form/cuts/course history, strokes-gained form.

In [9]:
from utils.model import build_current_week_rows

this_week = build_current_week_rows(ctx, dk, odds_current, tournament_config,
                                    allow_stale=test_mode)
this_week.head()

✅ odds coverage: 98% of DK field
⚠️ stats coverage: 63% of DK field
✅ SG form coverage: 97% of DK field


,PLAYER,SALARY,SGTTG_RANK,SGTTG,SGOTT_RANK,SGOTT,SGAPR_RANK,SGAPR,SGATG_RANK,SGATG,...,adj_form,PCT_FORM_SHRUNK,COURSE_HISTORY,adj_ch,PCT_CH_SHRUNK,SG_FORM,SG_ROUNDS_12M,SG_CH_SHRUNK,ODDS_SHARE,FIELD_SIZE
0,Jackson Koivun,10800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.11,0.3036,NaN,NaN,NaN,1.5299,30.0,NaN,0.054782,115
1,Michael Brennan,10200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,14.94,0.3933,1.0,1.44,0.3359,0.7221,101.0,2.6965,0.041892,115
2,Ben James,9900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,14.03,0.3405,NaN,NaN,NaN,1.4622,34.0,NaN,0.033913,115
3,Matt Wallace,9700,53.0,0.413,136.0,-0.163,81.0,0.093,4.0,0.483,...,16.81,0.4046,90.0,129.84,0.5606,0.7365,77.0,-1.0523,0.027391,115
4,Maverick McNealy,9600,54.0,0.390,61.0,0.185,48.0,0.280,120.0,-0.075,...,9.61,0.3313,90.0,129.84,0.5606,0.8034,82.0,-0.5523,0.033913,115


### Data Health Checks

A DK player missing **both** odds and SG history is almost always a name-map
miss (fix in `DK_PLAYER_NAME_MAP` / `PLAYER_NAME_MAP` in `utils/db_utils.py`),
not a real unknown. Missing one of the two is normal for rookies/alternates.

In [10]:
check = this_week[["PLAYER", "SALARY", "VEGAS_ODDS", "SG_FORM", "SGTTG"]].copy()
check["n_missing"] = check[["VEGAS_ODDS", "SG_FORM", "SGTTG"]].isna().sum(axis=1)
suspects = check[check["n_missing"] >= 2].sort_values("SALARY", ascending=False)
print(f"{len(suspects)} players missing 2+ key features:")
display(suspects)

# High-salary players with no odds match: usually an odds-side name typo
# (e.g. golfodds' "Piercescon Coody"). Fix with scope="player" below.
no_odds = check[check["VEGAS_ODDS"].isna() & (check["SALARY"] >= 7000)]
if len(no_odds):
    print()
    print(f"{len(no_odds)} players at $7k+ with no odds match (check the odds board for typos):")
    display(no_odds.sort_values("SALARY", ascending=False))

4 players missing 2+ key features:


,PLAYER,SALARY,VEGAS_ODDS,SG_FORM,SGTTG,n_missing
99,Boston Bracken,6200,1000.0,NaN,NaN,2
106,K.H. Lee,6100,NaN,-1.3559,NaN,2
111,David Lingmerth,6000,3000.0,NaN,NaN,2
114,Zach J. Johnson,6000,3000.0,NaN,NaN,2



1 players at $7k+ with no odds match (check the odds board for typos):


,PLAYER,SALARY,VEGAS_ODDS,SG_FORM,SGTTG,n_missing
37,Taylor Pendrith,7400,NaN,-0.1728,0.635,1


### Fix Name Mismatches

Maybe can remove this section since we hve the name_mappings.json file now.

Add mappings here — no need to edit `utils/db_utils.py`. They persist to
`data/name_mappings.json` and apply on every future run.

- `scope="dk"` — the DraftKings salary file spells it differently than the DB
  (e.g. `add_name_mapping("John Keefer", "Johnny Keefer", scope="dk")`)
- `scope="player"` — the odds scrape / PGA results spell it differently
  (e.g. `add_name_mapping("Johnny Keefer", "John Keefer")`)

The first argument is the WRONG (incoming) spelling, the second is the DB's
canonical spelling. After adding, re-run from the DraftKings Field cell.

In [10]:
from utils.db_utils import add_name_mapping, show_name_mappings

# "Wrong Name", "Correct Name"

# add_name_mapping("Matthew McCarty", "Matt McCarty", scope="dk")           # DK file side
# add_name_mapping("Wrong Name", "Canonical DB Name", scope="player")      # odds/results side
# add_name_mapping("Old Tourney Name", "Canonical Tourney Name", scope="tournament")    # odds/results side

# show_name_mappings()  # uncomment to print the full list of every mapping

### Column Glossary (derived features)

All rolling windows end the day **before** the tournament's ending date (equivalent
to its start morning — no other event finishes mid-week). "Percentile" always means
finish position within that event's field: **0 = won, 1 = finished last**; missed
cuts land together in the cut bucket (≈0.58 in a 156-man field).

**Market**
- `VEGAS_ODDS` — scraped fractional odds as a decimal (30/1 → 30.0). Display only; not a model input.
- `ODDS_SHARE` — the model's market feature. Implied win probability `1/(odds+1)`;
  unlisted players get half the field's smallest listed probability; then everything
  is divided by the field total so shares **sum to exactly 1**. Read it as the
  player's percent of the field's win equity (0.047 = 4.7%).

**Form**
- `SG_FORM` — strokes per round better (+) or worse (−) than the field average,
  built from every round in the last 2 years. A round's weight halves every 100
  days (a round from ~3 months ago counts half; ~6 months ago a quarter). Thin
  samples are shrunk toward 0 (field average). Guide: +1.5 elite-hot, +0.5 solid,
  0 tour-average, −1 struggling fringe. Missing (no rounds in 2 yrs) → filled at
  the training set's 25th percentile at fit time ("assume below average").
- `SG_ROUNDS_12M` — competitive rounds in the last 365 days; the sample size
  behind SG_FORM (82 = trust it, 6 = don't).
- `PCT_FORM_SHRUNK` — average finish percentile over the last 9 months, blended
  with **4 phantom perfectly-average (0.5) events** so small samples pull toward
  the middle. 0.37 = "typically finishes in the top ~37% of fields, sample-adjusted."
  Lower = better. Missing (inactive) → 0.8.
- `CUT_PERCENTAGE` — % of cuts made in the last 9 months.
- `CONSECUTIVE_CUTS` — current made-cut streak entering this week.
- `FEDEX_CUP_POINTS` — points earned in the last 9 months (rolling window, NOT the
  season-to-date total).
- `form_density` — FedEx points per event in that window.

**Course**
- `SG_CH_SHRUNK` — strokes gained per round AT this course over the past 7 years,
  shrunk toward 0 (field average) by 2 phantom average rounds. +1.5 = proven course
  horse, 0 = no history / average, negative = struggles here. Replaced the
  finish-position-based PCT_CH_SHRUNK (Stage 6: beat it in 4 of 5 eval seasons —
  strokes are a cleaner signal than finish positions).

**Context**
- `FIELD_SIZE` — number of players in the event. Calibration context for the model
  (what a given ODDS_SHARE means, how big the cut bucket is). Identical for every
  player in a given week, so it cancels out of the within-field ranking.

**Prior-season stats block** (`SGTTG`, `SGOTT`, `SGAPR`, `SGATG`, `SGP`, driving,
GIR, scrambling, OWGR + their `_RANK`s) — last season's full-year PGA Tour
aggregates. Individually near-zero importance (stale + redundant with odds/SG
form) but collectively still worth a small edge; kept as model inputs, mostly
dropped from the export.

## Train + Predict

A random-forest regressor is trained on every historical row (~50k) to predict
`FINISH_PCT`, then this week's field is scored and blended with the market.

**`MODEL_SCORE` = 1 − predicted finish percentile.** Pure model output (odds are
one of its inputs, but this number is whatever the forest says). 0.76 = "expected
to finish around the 24th percentile of this field." Magnitudes are meaningful —
a gap of 0.04 is a real skill gap, not just a rank step.

**`SCORE` — the optimizer column — is computed within this week's field:**

1. Rank all N players by `MODEL_SCORE` (1 = worst … N = best).
2. Rank all N players by `ODDS_SHARE`. Bookmakers quote tied prices (all the
   55/1s share one averaged rank) — the model is what breaks those ties.
3. Average the two ranks: `blended_rank = (model_rank + market_rank) / 2`.
4. Rescale: `SCORE = blended_rank / N` → range (0, 1], **1.0 = best in field**.

It's a blend of *ranks*, not values, because the two live on incomparable scales
(MODEL_SCORE ~0.4–0.8 vs ODDS_SHARE ~0.0002–0.05). The 50/50 weight is validated:
forward-chained eval scored the blend 6.49 hits@15 vs 6.41 model-only and 6.38
odds-only, with a flat optimum anywhere from 25% to 85% market weight.

**`LEVERAGE` = model_rank − market_rank.** Positive = the model likes the player
more than the market does (potential value play); negative = the market is higher
on him than the model. The biggest absolute values are this week's disagreements —
worth a look before locking a lineup.

In [11]:
from utils.model import train_and_score

scored, importances = train_and_score(training_df, this_week)

preview_cols = ["PLAYER", "SALARY", "P_TOP20", "SCORE", "MODEL_SCORE", "ODDS_SHARE", "LEVERAGE",
                "VEGAS_ODDS", "SG_FORM", "PCT_FORM_SHRUNK", "SG_CH_SHRUNK",
                "CUT_PERCENTAGE"]
scored[[c for c in preview_cols if c in scored.columns]].head(30)

,PLAYER,SALARY,P_TOP20,SCORE,MODEL_SCORE,ODDS_SHARE,LEVERAGE,VEGAS_ODDS,SG_FORM,PCT_FORM_SHRUNK,SG_CH_SHRUNK,CUT_PERCENTAGE
0,Jackson Koivun,10800,0.4943,1.0000,0.755868,0.054782,0.0,12.0,1.5299,0.3036,0.0000,87.5
1,Maverick McNealy,9600,0.4521,0.9848,0.716057,0.033913,1.5,20.0,0.8034,0.3313,-0.5523,90.5
2,Michael Brennan,10200,0.4493,0.9826,0.702093,0.041892,-2.0,16.0,0.7221,0.3933,2.6965,83.3
3,Ben James,9900,0.4307,0.9804,0.703491,0.033913,0.5,20.0,1.4622,0.3405,0.0000,88.9
4,Doug Ghim,9500,0.4037,0.9565,0.678765,0.027391,0.0,25.0,1.0177,0.3572,-0.0696,72.2
5,Matt Wallace,9700,0.3816,0.9609,0.683026,0.027391,1.0,25.0,0.7365,0.4046,-1.0523,70.6
6,Marco Penge,9400,0.3729,0.9522,0.671952,0.027391,-1.0,25.0,0.3787,0.4670,0.0000,62.5
7,Kevin Yu,9200,0.3483,0.9261,0.667456,0.019782,1.0,35.0,0.7641,0.4314,1.1965,60.0
8,Jackson Suber,9300,0.3460,0.9391,0.668785,0.022973,0.0,30.0,0.7093,0.3926,1.1965,68.4
9,Davis Thompson,9100,0.3452,0.9217,0.650037,0.019782,0.0,35.0,0.6105,0.3878,-1.3023,65.0


In [12]:
import plotly.express as px

imp_df = importances.head(20).reset_index()
imp_df.columns = ["Feature", "Importance"]
fig = px.bar(imp_df, x="Feature", y="Importance",
             title="Feature Importances (Percentile Regressor)",
             template="plotly_dark")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Export

Sorted by **P_TOP20** — the key column: a calibrated probability of finishing
top-20 (model blended with market-implied, magnitudes learned from a decade of
odds-vs-results). It is what the optimizer maximizes: **a lineup's sum of
P_TOP20 = its expected number of top-20 finishers.**

SCORE (rank blend, 1 = best) is kept as the pure-ordering view; LEVERAGE
(model rank − market rank) remains the value/fade tilt.

### Why `data/current_week_export.csv` is still written

Not for the retired Excel optimizer or the retired Streamlit app — the dashboard
replaced both. It keeps being written for two reasons:

1. **It is the tracked source the dashboard is rebuilt from.** `slate.js` is
   generated and gitignored, so on the other computer (or a fresh clone) the
   launcher cell below sees a missing/older slate and calls `rebuild_from_disk()`,
   which reads exactly this CSV — no odds scrape, no DK file, no retraining. It is
   also how staleness is detected: `slate_is_stale()` compares the slate's mtime
   against this file's. Delete the CSV and a pull can no longer reconstruct the week.
2. **It is the readable check on what the dashboard should be showing** — the frame
   printed below this cell is the same data the terminal renders, in the notebook
   where the numbers were produced.


In [13]:
export_cols = ["PLAYER", "SALARY", "P_TOP20", "SCORE", "MODEL_SCORE", "ODDS_SHARE", "LEVERAGE",
               "VEGAS_ODDS", "SG_FORM",
               "PCT_FORM_SHRUNK", "SG_CH_SHRUNK",
               "CUT_PERCENTAGE", "FEDEX_CUP_POINTS", "OWGR_RANK"]

export_df = scored[[c for c in export_cols if c in scored.columns]].copy()

round_map = {"P_TOP20": 3, "MODEL_SCORE": 4, "ODDS_SHARE": 4, "LEVERAGE": 1, "SG_FORM": 2,
             "PCT_FORM_SHRUNK": 3, "SG_CH_SHRUNK": 2,
             "CUT_PERCENTAGE": 1, "FEDEX_CUP_POINTS": 0, "OWGR_RANK": 0}
for col, nd in round_map.items():
    if col in export_df.columns:
        export_df[col] = export_df[col].round(nd)

export_df = export_df.sort_values("P_TOP20", ascending=False).reset_index(drop=True)
export_df.to_csv("data/current_week_export.csv", index=False)
print(f"✅ Exported {len(export_df)} players to data/current_week_export.csv")


# Log this week's forecast to the predictions table (feeds the Report Card cell
# and the dashboard's Prediction Tracker; re-running the same week is a no-op)
from utils.model import save_predictions, save_current_week_meta
save_predictions("data/golf.db", export_df, tournament_config, dry_run=test_mode)

# Refresh the current-week marker (data/current_week.json): the tournament,
# course and ending date that the slate is built against, and what
# rebuild_from_disk() reads when it reconstructs the slate on the other
# computer. Skipped in test_mode, whose config points at a past week — writing
# it then would leave the marker on a stale course.
if not test_mode:
    _meta = save_current_week_meta(tournament_config)
    print(f"✅ Week marker → {_meta['name']} @ {_meta['course']} ({_meta['ending_date']})")

# Publish the slate for the PGA Slate Terminal (dashboard/). Writes
# window.SLATE into dashboard/public/data/slate.js — a .js file rather than
# JSON because the built page must also work when opened by double-click, and
# file:// blocks fetch() of local files but permits <script src>.
# Skipped in test_mode so a dry-run against a past week can't leave the
# dashboard showing a stale field, same reasoning as the week marker above.
if not test_mode:
    import utils.dashboard
    importlib.reload(utils.dashboard)
    from utils.dashboard import export_dashboard
    export_dashboard(db_path, export_df, tournament_config)

export_df.head(15)

✅ Exported 115 players to data/current_week_export.csv
✅ Logged 115 predictions for Bank of Utah Championship (2026-10-04) — track record grows.
✅ Week marker → Bank of Utah Championship @ Black Desert Resort (2026-10-04)
✅ Dashboard slate → 115 players, 1082 KB
   90 with 2026 SG stats · 107 with 2026 OWGR · 13,841 rounds · 1143 tracked predictions (996 graded)
   SG rankings 942 players · Black Desert Resort: 127 players / 2 events
   → dashboard\public\data\slate.js
   → dashboard\dist\data\slate.js


,PLAYER,SALARY,P_TOP20,SCORE,MODEL_SCORE,ODDS_SHARE,LEVERAGE,VEGAS_ODDS,SG_FORM,PCT_FORM_SHRUNK,SG_CH_SHRUNK,CUT_PERCENTAGE,FEDEX_CUP_POINTS,OWGR_RANK
0,Jackson Koivun,10800,0.494,1.0000,0.7559,0.0548,0.0,12.0,1.53,0.304,0.00,87.5,730.0,178.0
1,Maverick McNealy,9600,0.452,0.9848,0.7161,0.0339,1.5,20.0,0.80,0.331,-0.55,90.5,1100.0,21.0
2,Michael Brennan,10200,0.449,0.9826,0.7021,0.0419,-2.0,16.0,0.72,0.393,2.70,83.3,1040.0,34.0
3,Ben James,9900,0.431,0.9804,0.7035,0.0339,0.5,20.0,1.46,0.340,0.00,88.9,665.0,1000.0
4,Doug Ghim,9500,0.404,0.9565,0.6788,0.0274,0.0,25.0,1.02,0.357,-0.07,72.2,428.0,148.0
5,Matt Wallace,9700,0.382,0.9609,0.6830,0.0274,1.0,25.0,0.74,0.405,-1.05,70.6,545.0,75.0
6,Marco Penge,9400,0.373,0.9522,0.6720,0.0274,-1.0,25.0,0.38,0.467,0.00,62.5,389.0,29.0
7,Kevin Yu,9200,0.348,0.9261,0.6675,0.0198,1.0,35.0,0.76,0.431,1.20,60.0,472.0,90.0
8,Jackson Suber,9300,0.346,0.9391,0.6688,0.0230,0.0,30.0,0.71,0.393,1.20,68.4,554.0,183.0
9,Davis Thompson,9100,0.345,0.9217,0.6500,0.0198,0.0,35.0,0.61,0.388,-1.30,65.0,459.0,92.0


## Open the Dashboard

The launcher. Serves the repo root on a background thread and opens the
**PGA Slate Terminal** with this week's field already loaded.

> ### ⚠️ Lineups only reach OneDrive while this kernel is alive
>
> The server is a thread **inside this notebook's Python kernel**, so whatever
> ends the kernel ends the writes. The dashboard itself keeps working — it falls
> back to the browser's own copy — but while the server is gone, lineup changes
> reach neither OneDrive nor the other computer.
>
> Restarting the kernel and closing VS Code always end it. **Closing this
> notebook tab sometimes does.** VS Code disposes the kernel when the notebook
> *document* closes, which is not the same event as the tab closing: a second
> editor group, an open diff view, or an extension holding the notebook can all
> keep the document — and the kernel — alive. So closing the tab is a coin flip
> rather than a rule, which is why lineups used to sync on some evenings and not
> others. Leaving the tab open while the dashboard is open is the way to be sure.
>
> **Rather than track any of that, read the lineup rail.** It shows an amber
> **"sync failed — browser copy kept"** exactly when writes are not landing, and
> stays silent when they are. Nothing is lost either way: reopen the dashboard
> with this cell running and the browser's copy is pushed to OneDrive
> automatically on load.

**Run this cell on its own.** It depends on no earlier cell, so opening the app on
a fresh kernel does not mean re-running the USER INPUTS cell or any of the weekly
workflow — and it rebuilds the slate first if it is missing or out of date.

**A `dashboard/src` edit rebuilds automatically, too.** Same idea as the slate check
above: this cell runs `npm run build` in `dashboard/` whenever `dist/index.html` is
older than the source (TypeScript/React files, `index.html`, `package.json`), so a
frontend change shows up just by re-running this cell — no separate build step to
remember. Needs `npm` on PATH and `dashboard/node_modules` already installed
(`npm install`, once); either missing, it says so and serves the last good build.

**Running it is always enough.** A server already running the current code is
reused; one running older code — an earlier run of this cell — is stopped and
replaced. Restarting the kernel is never the fix (it only discards the work
above; it is the *server* that goes stale, not the kernel). The one case it
cannot handle is a server inside a *different* kernel, and it says so rather than
serving stale behaviour.

Read the last line of the output:

| | |
|---|---|
| ⏳ `rebuilding dist…` | dashboard/src had changed — building the new bundle first |
| ✅ `Serving …` | fresh server started |
| ✅ `already running … up to date` | nothing needed to change |
| ♻️ `Replacing the server …` | stale one was swapped out, now current |
| ⚠️ … | nothing was changed — the line says what to do |

Served from the repo root rather than from `dashboard/dist` so the Results Browser
can read `data/golf.db` in place — copying the 20 MB database into `dashboard/`
each week would add a 20 MB blob to git history every time.

**Lineups** autosave to `current.json` under `Fantasy Golf/Lineup_Optimizer` in
**personal** OneDrive — outside the repo, so OneDrive carries them between
computers on its own and the working tree stays clean. Nothing to commit or pull;
just open the dashboard on the other machine.

Personal specifically: on a machine signed into a work account, `%OneDrive%` is
the *work* tenant, which will not sync onto a home computer — so the launcher
looks for the consumer account (registry, then `%OneDriveConsumer%`, then
`%USERPROFILE%\OneDrive`) and never writes into an `OneDrive - <Org>` folder. If a
machine has no personal OneDrive at all, lineups fall back to `data/lineups` and
the launcher prints a ⚠️ saying they will not reach the other computer.


In [14]:
# Standalone on purpose: this cell depends on no other cell in the notebook, so
# a fresh kernel can run THIS ONE to open the dashboard without first running
# the USER INPUTS cell or any of the weekly workflow.
#
# ⚠️ Lineups only reach OneDrive while THIS KERNEL is alive — the server below
# is a thread inside it. Restarting the kernel and closing VS Code always end
# it; closing the notebook tab sometimes does (VS Code disposes the kernel when
# the notebook DOCUMENT closes, which the tab closing does not always trigger).
# Keeping the tab open while the dashboard is open is the way to be sure. Either
# way the rail turns amber ("sync failed") whenever writes are not landing, and
# the browser's copy is pushed to OneDrive on the next load with a live server.
import importlib
import utils.dashboard
importlib.reload(utils.dashboard)
from utils.dashboard import (serve_dashboard, resolve_lineup_dir,
                             slate_is_stale, rebuild_from_disk)

# Lineups (locks, excludes, picks, saved lineups) autosave here:
#
#     <personal OneDrive>\Fantasy Golf\Lineup_Optimizer\current.json
#
# Outside the repo, so OneDrive carries them between computers on its own —
# nothing to commit or pull, and the working tree stays clean. The PERSONAL
# OneDrive root is found per machine, so the same default is correct on both
# computers despite the different usernames.
#
# Personal specifically, not %OneDrive%: on the work laptop that variable points
# at the Emerson tenant, which refuses to sync onto a home computer — lineups
# saved there are unreachable from the other machine, which defeats the point.
# If a machine has no personal OneDrive, the launcher says so rather than
# quietly writing somewhere that never syncs.
#
# Leave LINEUP_DIR as None for that. Set a path only if the folder ever moves.
LINEUP_DIR = None

# slate.js is generated weekly and gitignored, so on a cold start it may be
# missing (fresh clone) or older than an export a pull brought in — either way
# the dashboard would open on the wrong field with nothing to say so. Rebuilt
# from files already on disk: no odds scrape, no DraftKings file, no training.
if slate_is_stale():
    print("⏳ Slate missing or out of date — rebuilding…")
    rebuild_from_disk()

print("lineups →", resolve_lineup_dir(LINEUP_DIR))

# Running this cell is always enough. A server already running this code is
# reused; one running older code — an earlier run of this cell — is shut down
# and replaced. So there is no state where the dashboard is quietly serving
# stale behaviour: either the ✅ line below says it is current, or a ⚠️ line
# says what to do. Nothing to restart by hand, and in particular no need to
# restart the kernel (which would only throw away the dataframes above; it is
# the SERVER that goes stale, not the kernel).
serve_dashboard(lineup_dir=LINEUP_DIR)


lineups → C:\Users\rmiller\OneDrive\Fantasy Golf\Lineup_Optimizer
⏳ Dashboard source changed — rebuilding dist…
✅ Dashboard rebuilt.
✅ Serving c:\Users\rmiller\Documents\Github\pga-dk on port 8765
   http://localhost:8765/dashboard/dist/index.html?v=1790703210
📁 Lineups: C:\Users\rmiller\OneDrive\Fantasy Golf\Lineup_Optimizer\current.json


'http://localhost:8765/dashboard/dist/index.html?v=1790703210'